# GPT-2 Text Generation - Generative AI Lab Assignment

In [1]:
# Install required library
!pip install transformers -q

import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
print('Setup complete!')

Using device: cpu
Setup complete!


In [2]:
MODEL_NAME = 'gpt2'

tokenizer = GPT2Tokenizer.from_pretrained(MODEL_NAME)
model = GPT2LMHeadModel.from_pretrained(MODEL_NAME)

tokenizer.pad_token = tokenizer.eos_token

model = model.to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f'Model: {MODEL_NAME}')
print(f'Total parameters: {total_params:,}')
print('GPT-2 loaded successfully!')

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Model: gpt2
Total parameters: 124,439,808
GPT-2 loaded successfully!


In [3]:
def generate_text(prompt, max_new_tokens=100, top_k=40, temperature=1.0):
    inputs = tokenizer.encode(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_k=top_k,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id
        )

    generated = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return generated

test_output = generate_text('The future of AI is', max_new_tokens=50, top_k=40, temperature=1.0)
print(test_output)

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


The future of AI is open for us."

So far, the two countries have worked out how to best exploit the technologies for their own gain. Japan recently gave up a portion of its "high-end mobile games royalties" to Sony, and it's unlikely it


In [4]:
templates = {
    'Instructional': 'Write a short paragraph about: {topic}',
    'Role-based'   : 'You are a teacher. Explain {topic} in simple terms.',
    'Creative'     : 'Write a futuristic story opening where {topic} changes the world.',
}

topics = ['artificial intelligence', 'climate change', 'space exploration']

results = []
for topic in topics:
    for template_name, template in templates.items():
        prompt = template.format(topic=topic)
        output = generate_text(prompt, max_new_tokens=100, top_k=40, temperature=1.0)
        results.append({
            'Topic'   : topic,
            'Template': template_name,
            'Prompt'  : prompt,
            'Output'  : output[:200]
        })
        print(f'[{template_name}] Topic: {topic}')
        print(output[:120])
        print()

[Instructional] Topic: artificial intelligence
Write a short paragraph about: artificial intelligence (ASI) The best way to understand AI is to go into it and say it's

[Role-based] Topic: artificial intelligence
You are a teacher. Explain artificial intelligence in simple terms. Tell us your thoughts on the world's most important 

[Creative] Topic: artificial intelligence
Write a futuristic story opening where artificial intelligence changes the world.

We need heroes, we don't need to be r

[Instructional] Topic: climate change
Write a short paragraph about: climate change and what you can do to curb it?

The climate change discussion

It was lon

[Role-based] Topic: climate change
You are a teacher. Explain climate change in simple terms. It cannot have been fixed for 20,000 years or even 10,000 yea

[Creative] Topic: climate change
Write a futuristic story opening where climate change changes the world.

[Instructional] Topic: space exploration
Write a short paragraph about: space

In [5]:
fixed_prompt = 'The future of education is'
top_k_values   = [10, 40, 80]
temp_values    = [0.7, 1.0, 1.3]

sweep_results = []

for top_k in top_k_values:
    for temp in temp_values:
        output = generate_text(
            fixed_prompt,
            max_new_tokens=80,
            top_k=top_k,
            temperature=temp
        )
        sweep_results.append({
            'top_k': top_k,
            'temperature': temp,
            'output': output[len(fixed_prompt):].strip()
        })
        print(f'top_k={top_k}, temp={temp}')
        print(output[:120])
        print()

top_k=10, temp=0.7
The future of education is a very important issue for all of us," she said.

"There's a lot of money to be spent on scho

top_k=10, temp=1.0
The future of education is at stake in the next two years because we can't afford to continue the current system without

top_k=10, temp=1.3
The future of education is in doubt. It is impossible to have the best education in any country," he wrote in a post on 

top_k=40, temp=0.7
The future of education is, of course, a question of education. It seems that we might be able to achieve it, we might h

top_k=40, temp=1.0
The future of education is about knowing how to succeed: we all know it already.

One is not necessarily a genius; he ma

top_k=40, temp=1.3
The future of education is not determined by the government's actions," Sallis said when I spoke on April 23, 2013, abou

top_k=80, temp=0.7
The future of education is up in the air; the future of culture is up in the air; and the future of the economy is up in

top_k=80, tem

In [6]:
df = pd.DataFrame(sweep_results)
df['length'] = df['output'].apply(len)
df['repetition_rate'] = df['output'].apply(
    lambda x: len(x.split()) - len(set(x.split()))
)

print(df[['top_k', 'temperature', 'length', 'repetition_rate']])
df.to_csv('generation_results.csv', index=False)
print('Results saved to generation_results.csv')

   top_k  temperature  length  repetition_rate
0     10          0.7     329               34
1     10          1.0     399               17
2     10          1.3     367               13
3     40          0.7     352               17
4     40          1.0     348               21
5     40          1.3     328               10
6     80          0.7     364               29
7     80          1.0     353                6
8     80          1.3     434               13
Results saved to generation_results.csv


In [7]:
def generate_text_topp(prompt, max_new_tokens=100, top_p=0.9, temperature=1.0):
    inputs = tokenizer.encode(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            top_p=top_p,
            temperature=temperature,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

prompt = 'Artificial intelligence will'
print('--- Top-k (k=40) ---')
print(generate_text(prompt, top_k=40, temperature=1.0)[:200])

print('\n--- Top-p (p=0.9) ---')
print(generate_text_topp(prompt, top_p=0.9, temperature=1.0)[:200])

--- Top-k (k=40) ---
Artificial intelligence will take care of the job – and it will continue to do so.

Now here is an example — imagine a business with 1 million users. If one hundred million users were willing to pay a

--- Top-p (p=0.9) ---
Artificial intelligence will play a crucial role in providing real-time control of the Internet and the way we interact with the world. Our technology is changing the way we communicate, our lives, ou
